In [9]:
import pickle
import pandas as pd

with open('name_to_fips.pkl', 'rb') as f:
    name_to_fips = pickle.load(f)

with open('us_dict.pkl', 'rb') as f:
    us_dict = pickle.load(f)

with open('type_to_data.pkl', 'rb') as f:
    type_to_data = pickle.load(f)

with open('state_to_county_data.pkl', 'rb') as f:
    state_to_county_data = pickle.load(f)

In [10]:
def check_invalids(given_list, state = True):
    """
    Checks if list of fips is valid by comparing
    """
    fips_codes = name_to_fips.values()
    if state: # checking state fips codes
        invalid_states = [elem for elem in given_list if elem not in fips_codes]
    else:
        invalid_states = [elem for elem in given_list if elem not in fips_codes]


    if invalid_states: # not empty
        raise ValueError(f"Invalid state FIPS codes: {', '.join(invalid_states)}")



def retrieve_data():
    """
    This function works by allowing the user to give different inputs:
    for a state, county, and DER type. Each option can input one, multiple
    by separating with a comma or all by typing "all". If choosing all after choosing
    multiple states then that will give all the counties within those states. If you choose
    one certain DER type then that will give you that one DER type from all the counties
    chosen.

    Returns:
        The requested data inputted using the terminal or the dropdown.
    """
    # STATES !!
    # first get the state
    state_inputs = input("\nEnter the state FIPS codes or names (comma-separated, or type 'exit' to quit): or type 'all' to select all states: ").strip()
    print("state inputs", state_inputs)
    if state_inputs.lower() == 'exit':
        return "Exiting..."
    # parses the given input into a list and makes the names into fips codes
    elif state_inputs.lower() != 'all':
        state_fips_list = [state.strip()
            if state.strip().isnumeric()
            else name_to_fips.get(state.strip().lower(), 0)
            for state in state_inputs.split(",")]
        print("state fips", state_fips_list) # debugging
        check_invalids(state_fips_list) # check if given invalid fips code
    else: # for choosing all states
        state_fips_list = list(us_dict.keys())

    # COUNTIES !!
    county_input = input(
        "Enter county FIPS codes or names (comma-separated, or type 'all' to select all counties): ").strip()
    # parses the given input into a list and makes the names into fips codes
    if county_input.lower() != 'all':
        county_list = [county.strip().lower()
            if county.strip().lower().isnumeric()
            else name_to_fips.get(county.strip().lower(), 0)
            for county in county_input.split(",")]
        check_invalids(county_list, False) # check if given invalid fips code
    else: # for all counties
        county_list = []
        for state in state_fips_list:
            county_list += us_dict[state]


    # DER TYPES !!!
    der_types_input = input(
            "Enter DER types (comma-separated, or type 'all' to select all available types): ").strip()
    if der_types_input.lower() != 'all':
        der_types = [der.strip().lower()
                     for der in der_types_input.split(",")
                     if der.strip.lower() in type_to_data]
    else: # if inputted all
        der_types = list(type_to_data.keys())


    # filters data based on the inputs
    filtered_data = []
    for state in state_fips_list:
        for county in county_list:
            for der_type in der_types:
                # example structure: data[state][county][der_type]
                if state in state_to_county_data and county in state_to_county_data[state] and der_type in state_to_county_data[state][county]:
                    filtered_data.append({
                        "state": state,
                        "county": county,
                        "der_type": der_type,
                        "data": state_to_county_data[state][county][der_type]
                    })

    # If no data is found, inform the user
    if not filtered_data:
        return "No data found for the selected criteria."

    # Return or display the results
    print("\nFiltered Data:")
    export_data(filtered_data)

    return filtered_data

retrieve_data()

state inputs all

Filtered Data:


[{'state': '4000',
  'county': '4013',
  'der_type': 'powerplants',
  'data': {'S': [{'objectid': 2820,
     'cecplantid': 'S0254',
     'plantname': 'Mesquite Solar 1 (AZ)',
     'retired plant': 0,
     'operator company': 'RWE Clean Energy',
     'county': 'Maricopa',
     'capacity_latest': 165.0,
     'units': 'MS1',
     'prienergysource': 'SUN',
     'startdate': '1/1/2014 12:00:00 AM',
     'cec_jurisdictional': 0.0,
     'x': -112.923873603,
     'y': 33.343024728},
    {'objectid': 2821,
     'cecplantid': 'S0292',
     'plantname': 'Arlington Valley Solar Energy II (AZ)',
     'retired plant': 0,
     'operator company': 'LS Power',
     'county': 'Maricopa',
     'capacity_latest': 129.0,
     'units': '5912038',
     'prienergysource': 'SUN',
     'startdate': '11/5/2013 12:00:00 AM',
     'cec_jurisdictional': 0.0,
     'x': -112.833748578,
     'y': 33.3020589710001},
    {'objectid': 2822,
     'cecplantid': 'S0561',
     'plantname': 'Mesquite Solar 2 (AZ)',
     'reti

In [11]:
def export_data(der_data):
    """
    Exports dictionary as either CSV or XLSX

    Arguments:
        der_data (dict): contains the mappings of state to county to der type to data
    Returns:
        None
    """
    df = pd.DataFrame(der_data)

    # Ask if the user wants to export the data
    export_choice = input("Do you want to export this data? (yes/no): ").strip().lower()
    if export_choice == 'yes':
        file_type = input("Export as (1) CSV or (2) Excel? Enter 1 or 2: ").strip()
        file_name = input("Enter the file name (without extension): ").strip()

        try:
            if file_type == "1":
                df.to_csv(f"{file_name}.csv", index=False)
                print(f"Data exported to {file_name}.csv")
            elif file_type == "2":
                df.to_excel(f"{file_name}.xlsx", index=False)
                print(f"Data exported to {file_name}.xlsx")
            else:
                print("Invalid choice. No file was saved.")
        except Exception as e:
            print(f"An error occurred while exporting the data: {e}")
    else:
        return df